# 🧪 Lab: Multi-Agent Workflow with LangChain + OpenAI in Databricks

### 🎯 Objective

In this lab, you'll build and run a **multi-agent system** that automates part of an insurance claim processing pipeline using LangChain and OpenAI. This hands-on workflow demonstrates how to:

- Chain multiple specialized tools using LangChain agents
- Use OpenAI to reason through structured task steps
- Coordinate decisions like validation, risk assessment, and resolution

This mirrors real-world architectures used in industries like finance, healthcare, and insurance.

---

### 📍 Scenario

You're working with a company that handles auto insurance claims. Rather than using one large AI model to handle all tasks, you’ll create a pipeline of **four agents**, each with a distinct role:

1. **Document Agent** – Extracts structured data from a claim report  
2. **Policy Agent** – Validates whether the policy is active and eligible  
3. **Assessment Agent** – Evaluates if the claim should be auto-approved or reviewed  
4. **Resolution Agent** – Generates the final output (approval or manual review)

By the end of this lab, you’ll simulate this pipeline in Databricks using **LangChain + OpenAI**, connected through a zero-shot reasoning agent.

---


### 🔧 Step 1: Install Required Packages

To work with LangChain agents and OpenAI, you need to install the appropriate versions of the required libraries. This command ensures compatibility with LangChain's classic structure (`langchain.llms.OpenAI`), which is recommended for Databricks environments.

```python
%pip install --upgrade openai "langchain<0.1"


In [0]:

%pip install --upgrade openai "langchain<0.1"



### 🔄 Step 2: Restart the Python Kernel

After installing or upgrading packages in Databricks, it’s important to restart the Python runtime so your notebook picks up the new dependencies cleanly.

```python
%restart_python


In [0]:
%restart_python 

### 🔑 Step 3: Get and Set Your OpenAI API Key

To interact with GPT models from Databricks, you'll need an OpenAI API key.

---

#### 📌 How to Get Your OpenAI API Key

1. Go to [https://platform.openai.com/account/api-keys](https://platform.openai.com/account/api-keys)
2. Log in or create an OpenAI account
3. Click **“Create new secret key”**
4. Copy the generated key (it starts with `sk-...`)
5. Keep it safe — you won’t see it again!

---

#### 🔐 How to Set the Key in Databricks (Development Only)

Paste your key in the cell below using `os.environ`:

```python
import os

# ⚠️ Development Only — for production, use Databricks Secrets
os.environ["OPENAI_API_KEY"] = "sk-..."  # 🔁 Replace this with your real key


In [0]:
import os
os.environ["OPENAI_API_KEY"] = "XXX"  # Replace with your real key


### 🗃️ Step 4: Load Sample Claims Data into a Spark Table

This step creates a simulated insurance claims dataset using PySpark. It includes three fictional claims.The dataset includes fields like:

- `claim_id`: Unique claim identifier
- `claimant_name`: Person who filed the claim
- `damage_description`: Text summary of the incident
- `estimated_damage`: Approximate cost of repair
- `policy_id`: Insurance policy reference

In this step, you’ll simulate an insurance claims table using Spark. This allows your agents to query real data during their decision-making.



Once loaded, the data is registered as a temporary SQL view called `claims`. This allows agents and tools in later steps to query real-time structured data directly from Spark.

> 🧪 You'll use this view as the foundation for extracting claim details, validating policies, and estimating damage throughout the multi-agent workflow.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = SparkSession.builder.getOrCreate()

# Define schema and sample data
schema = StructType([
    StructField("claim_id", StringType(), True),
    StructField("claimant_name", StringType(), True),
    StructField("damage_description", StringType(), True),
    StructField("estimated_damage", DoubleType(), True),
    StructField("policy_id", StringType(), True)
])

data = [
    ("C101", "Alice Jones", "Rear-end collision, bumper damage", 2200.0, "P1001"),
    ("C102", "David Kim", "Broken windshield and headlight", 850.0, "P1002"),
    ("C103", "Maria Patel", "Side door dent and paint scratch", 1200.0, "P1003")
]

df = spark.createDataFrame(data, schema)
df.createOrReplaceTempView("claims")

display(df)


### 🧠 Step 5: Use `PromptTemplate` and `LLMChain` to Assess Damage

In this step, you introduce two key LangChain concepts:

- `PromptTemplate`: A reusable prompt that defines how to phrase your request to the LLM
- `LLMChain`: A logic wrapper that combines the LLM and the prompt to produce structured outputs

Instead of hardcoding business rules in Python, you're now asking the LLM to decide whether a claim should be auto-approved or manually reviewed based on the estimated cost.

---

#### 🔍 What This Step Does:

- **Initializes the LLM** using OpenAI with `temperature=0` for deterministic outputs
- **Creates a prompt** that asks the model to evaluate the repair estimate
- **Wraps the prompt in an `LLMChain`** so it can be used like a function
- **Defines the tool** `assess_damage_llm()` which extracts a dollar amount from the text and passes it to the LLM chain

> 🧩 This makes your system more flexible and realistic — and teaches a pattern that's directly tested in the certification exam.


In [0]:
# Initialize OpenAI LLM
from langchain.llms import OpenAI
llm = OpenAI(temperature=0)

# Define LLM-based PromptTemplate for Assessing Damage
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import re

# Create a reusable prompt
damage_prompt = PromptTemplate.from_template(
    "The estimated damage cost is ${amount}. Should this claim be auto-approved or manually reviewed? Respond in one sentence."
)

# Create a chain that uses the prompt and LLM
damage_chain = LLMChain(llm=llm, prompt=damage_prompt)

# Define the tool logic using LLMChain
def assess_damage_llm(text):
    match = re.search(r"\$?(\d+[.,]?\d*)", text)
    if not match:
        return "Could not extract a valid dollar amount."
    return damage_chain.run({"amount": match.group(1)})


### 🛠️ Step 6: Define Tool Functions for Multi-Agent Workflow

This step defines four custom Python functions that act as **tools** for your LangChain agent. Each tool performs a distinct part of the insurance claim process:

- **`extract_claim_details(claim_id)`**  
  Queries the Spark `claims` table and returns structured information like the claimant’s name, damage details, cost estimate, and policy ID.

- **`validate_policy(policy_id)`**  
  Simulates a backend check against a list of valid policy IDs. Returns whether the given policy is active and valid.

- **`assess_damage(text)`**  
  Uses regular expressions to extract the dollar amount from the input. If the damage exceeds \$2,000, it flags the claim for manual review; otherwise, it auto-approves it.

- **`finalize_resolution(assessment)`**  
  Accepts the final decision string and formats it into a resolution message to be delivered to the user.

> 🧩 These functions will be wrapped as LangChain tools in the next step, allowing a reasoning agent to call them dynamically based on the task prompt.


In [0]:
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.llms import OpenAI
# Claim lookup tool
def extract_claim_details(claim_id):
    claim = spark.sql(f"SELECT * FROM claims WHERE claim_id = '{claim_id}'").first()
    if not claim:
        return f"No claim found for ID {claim_id}"
    return f"Claimant: {claim.claimant_name}, Damage: {claim.damage_description}, Estimate: ${claim.estimated_damage}, Policy#: {claim.policy_id}"

def validate_policy(policy_id):
    # Simulate valid policies
    valid_policies = {"P1001", "P1002", "P1003"}
    return f"Policy {policy_id} is valid." if policy_id in valid_policies else f"Policy {policy_id} is NOT valid."

 

def finalize_resolution(assessment):
    return f"Final decision: {assessment}"

import re
def assess_damage_llm(text):
    amount = re.search(r"\$?(\d+[.,]?\d*)", text)
    if not amount:
        return "Could not extract a valid dollar amount."
    return damage_chain.run({"amount": amount.group(1)})


### 🤖 Step 7: Register Tools and Initialize a Reasoning Agent

Now that your helper functions are ready, you’ll convert them into LangChain-compatible tools and initialize a **zero-shot reasoning agent** using OpenAI.

Here’s what happens in this step:

- **Tools Registration**:  
  - `ExtractClaim`: Queries the Spark table to retrieve claim details.  
  - `ValidatePolicy`: Checks if the policy is valid.  
  - `AssessDamage`: Decides if the claim should be auto-approved or manually reviewed.  
  - `FinalizeResolution`: Formats the outcome into a final decision message.

- **Agent Setup**:  
  The `OpenAI` LLM is wrapped with LangChain’s `initialize_agent()` function using the `ZERO_SHOT_REACT_DESCRIPTION` mode. This enables the agent to:
  - Read the user’s input  
  - Decide which tools to use  
  - Call them step-by-step with context  
  - Combine outputs into a meaningful answer

- **Execution**:  
  The final line runs the full reasoning loop on a real claim (`C101`) by issuing a prompt like:
  
  > `"Process claim C101"`

> 🧠 This is where reasoning meets automation — the agent decides what to do, in what order, and how to assemble the final result.


In [0]:
tools = [
    Tool(name="ExtractClaim", func=lambda x: extract_claim_details(x), description="Look up claim details from Spark."),
    Tool(name="ValidatePolicy", func=lambda x: validate_policy(x), description="Check policy validity."),
    Tool(name="AssessDamage", func=assess_damage_llm, description="Estimate damage using LLM and prompt."),
    Tool(name="FinalizeResolution", func=finalize_resolution, description="Output final claim decision.")
    
]

llm = OpenAI(temperature=0)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent using real data
result = agent.run("Process claim C101")
print(result)


### ✅ Interpreting the Agent Output

The agent has successfully processed claim **C101** using a multi-step reasoning approach. Here's what happened at each stage:

---

#### 🧠 Reasoning Breakdown

- **User Prompt**:  
  *“Process claim C101”*

- **Step 1 – ExtractClaim**  
  - **Input**: `C101`  
  - **Output**: Claimant: *Alice Jones*, Damage: *Rear-end collision*, Estimate: **$2200.0**, Policy#: **P1001**

- **Step 2 – ValidatePolicy**  
  - **Input**: `P1001`  
  - **Output**: *Policy is valid*

- **Step 3 – AssessDamage**  
  - **Input**: `$2200.0`  
  - **Output**: *Manual review required due to high cost*

- **Step 4 – FinalizeResolution**  
  - **Input**: *Manual review*  
  - **Output**: *Final decision: Manual review*

---

#### 🟢 Final Outcome

The agent evaluated all necessary inputs and determined:

> **The final decision for claim C101 is that a manual review is required due to the high cost of the damage estimate.**

---

> 💡 This output shows how LangChain agents combine tool outputs, internal reasoning, and natural language explanation — all from a single prompt.
